In [1]:
# ============================================================================
# 06_validation_vs_crsp.ipynb
# Validate the free-data accrual results against the survivorship-corrected,
# CRSP-based Kenneth French accrual-sorted portfolios. This is the paper's
# central validity check (referee concern): the SIGN of the anomaly should
# agree even if the free-data MAGNITUDE is inflated.
# ============================================================================

In [2]:
# --- Imports --------------------------------------------------------------
import urllib.request, io, zipfile
import pandas as pd, numpy as np
DATA_DIR, TAB_DIR = "../data", "../results/tables"

In [3]:
# --- Download the CRSP-based Kenneth French accrual portfolios -------------
url = "https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/Portfolios_Formed_on_AC_CSV.zip"
raw = urllib.request.urlopen(
    urllib.request.Request(url, headers={"User-Agent":"Mozilla/5.0"}), timeout=30).read()
txt = zipfile.ZipFile(io.BytesIO(raw)).read("Portfolios_Formed_on_AC.csv").decode("latin-1")
lines = txt.splitlines()

In [4]:
# --- Parse the value-weighted monthly block -------------------------------
start=None; header=None
for i,line in enumerate(lines):
    if "Lo 20" in line and "Qnt 2" in line:
        header=[x.strip() for x in line.split(",")]; start=i+1; break
cols=header[1:]
data=[]
for line in lines[start:]:
    p=[x.strip() for x in line.split(",")]
    if len(p)>=2 and p[0].isdigit() and len(p[0])==6:
        try: data.append([p[0]]+[float(x) for x in p[1:len(cols)+1]])
        except: pass
    elif data and line.strip()=="":
        break
df=pd.DataFrame(data, columns=["ym"]+cols)
df["ym"]=pd.to_datetime(df["ym"], format="%Y%m")
win=df[(df["ym"]>="2013-07-01")&(df["ym"]<="2021-06-30")]
print("CRSP window months:", len(win))

CRSP window months: 96


In [5]:
# --- Compare hedge sign and magnitude -------------------------------------
qcols=["Lo 20","Qnt 2","Qnt 3","Qnt 4","Hi 20"]
print("CRSP mean monthly returns by accrual quintile (%):")
print(win[qcols].mean().round(3).to_string())
hedge = win["Lo 20"] - win["Hi 20"]
t = hedge.mean()/hedge.std()*np.sqrt(len(hedge))
print(f"\nCRSP low-minus-high hedge: {hedge.mean():.3f}%/mo, t={t:.2f}")

# Free-data hedge for comparison
free = pd.read_csv(f"{TAB_DIR}/tacc_portfolio_returns.csv", index_col=0)
fm = free[[f"P{i}" for i in range(1,6)]].mean()*100
free_hedge = fm["P5"] - fm["P1"]  # P5=low accrual, P1=high
print(f"Free-data low-minus-high hedge: {free_hedge:.3f}%/mo (Carhart t=3.08 in notebook 03)")
print("\nConclusion: sign agrees (both positive, Sloan direction);",
      "free-data magnitude is ~an order of magnitude larger (survivorship + micro-cap).")

CRSP mean monthly returns by accrual quintile (%):
Lo 20    1.563
Qnt 2    1.468
Qnt 3    1.146
Qnt 4    1.202
Hi 20    1.405

CRSP low-minus-high hedge: 0.158%/mo, t=0.81
Free-data low-minus-high hedge: 2.260%/mo (Carhart t=3.08 in notebook 03)

Conclusion: sign agrees (both positive, Sloan direction); free-data magnitude is ~an order of magnitude larger (survivorship + micro-cap).
